# resolveWavesHayashi

Used by `mjo_wavenum_freq_season`

- "Reorder the complex coefficients returned by cfftf to resolve the progressive and retrogressive waves"
- `cfftf`: Performs a forward complex discrete fourier transform of a real periodic sequence

## cfftf(xr, xi)
- xr: a variable containing one or more periodic sequences to be transformed (real part of a complex periodic sequence), for a multi-dimensioned array the rightmost dimension will be transformed
- xi: a variable containing one or more periodic sequences to be transformed

In [2]:
import numpy as np

In [3]:
def cfftf(data):
    fft_data = np.fft.fft(data)
    return fft_data

In [4]:
# example data

x = [1002, 1017, 1018, 1020, 1018, 1027,
      1028, 1030, 1012, 1012, 982, 1012,
      1001, 996, 995, 1011, 1027, 1025,
      1030, 1016, 996, 1006, 1002, 982]

ncl_real_outputs = [24265, 16.06125, -161.8083,
                   26, 39.5, -64.78089, 1, 
                   -32.69729, 32.5, 26, -4.191688,
                   35.41693, -43, 35.41693, -4.191688,
                   26, 32.5, -32.69729, 1, -64.78089, 
                   39.5, 26, -161.8083, 16.06125]

cf = cfftf(x)
cf_real_outputs = []
for i, c in enumerate(cf):
    print(f"x = {x[i]}, \n\t{abs(ncl_real_outputs[i] - c.real) < 0.0001}, diff = {abs(ncl_real_outputs[i] - c.real)}")

x = 1002, 
	True, diff = 0.0
x = 1017, 
	True, diff = 1.7284586419918924e-06
x = 1018, 
	True, diff = 1.1744383897394073e-05
x = 1020, 
	True, diff = 0.0
x = 1018, 
	True, diff = 0.0
x = 1027, 
	True, diff = 4.027682010132594e-06
x = 1028, 
	True, diff = 7.105427357601002e-15
x = 1030, 
	True, diff = 6.97779704239565e-07
x = 1012, 
	True, diff = 0.0
x = 1012, 
	True, diff = 7.105427357601002e-15
x = 982, 
	True, diff = 2.556160838551591e-07
x = 1012, 
	True, diff = 2.9970030723802665e-06
x = 1001, 
	True, diff = 7.105427357601002e-15
x = 996, 
	True, diff = 2.997003079485694e-06
x = 995, 
	True, diff = 2.5561608563151594e-07
x = 1011, 
	True, diff = 7.105427357601002e-15
x = 1027, 
	True, diff = 0.0
x = 1025, 
	True, diff = 6.97779704239565e-07
x = 1030, 
	True, diff = 2.220446049250313e-15
x = 1016, 
	True, diff = 4.027682010132594e-06
x = 996, 
	True, diff = 7.105427357601002e-15
x = 1006, 
	True, diff = 3.552713678800501e-15
x = 1002, 
	True, diff = 1.1744383897394073e-05
x = 982, 


`resolveWaveHayashi` used by wavenumber-frequency spectral analysis in xarray
- "Resolve the complex coefficients returned by cfftf to resolve the progressive and retrogressive waves"
- `x`: three-dimensional array. The leftmost dimension contains the real and imaginary coefficients, the middle dimension referes to longitude, the rightmost dimension refers to time
- `window`: number od ays per season/segment
- `spd`: samples per day, spd=1 is daily data, spd=2 for 12-hourly data



In [38]:
import xarray as xr

In [70]:
# example data
time = np.arange(10+1)
lon = np.linspace(0, 360, 5)

# wave moves east (progressive)
wave = np.sin(2  * (0.1 * time[:, None] - 0.05 * lon[None, :]))
example_data = xr.DataArray(wave, dims=('time', 'lon'),
                            coords={'time': time, 'lon': lon})
print(example_data)

<xarray.DataArray (time: 11, lon: 5)> Size: 440B
array([[ 0.        , -0.41211849,  0.75098725, -0.95637593,  0.99177885],
       [ 0.19866933, -0.58491719,  0.86720218, -0.9953511 ,  0.94658685],
       [ 0.38941834, -0.7343971 ,  0.9488445 , -0.99464477,  0.86365741],
       [ 0.56464247, -0.85459891,  0.99265938, -0.95428509,  0.74629668],
       [ 0.71735609, -0.94073056,  0.99690007, -0.87588108,  0.59918345],
       [ 0.84147098, -0.98935825,  0.96139749, -0.76255845,  0.42818267],
       [ 0.93203909, -0.99854335,  0.88756703, -0.61883502,  0.2401116 ],
       [ 0.98544973, -0.96791967,  0.77835208, -0.45044059,  0.04246803],
       [ 0.9995736 , -0.8987081 ,  0.63810668, -0.26408852, -0.1568686 ],
       [ 0.97384763, -0.79366786,  0.47242199, -0.06720807, -0.34995137],
       [ 0.90929743, -0.6569866 ,  0.28790332,  0.13235175, -0.52908269]])
Coordinates:
  * time     (time) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lon      (lon) float64 40B 0.0 90.0 180.0 270.0 360.0


In [17]:
def resolveWavesHayashi(var_fft: xr.DataArray,
                        nDayWin: int, 
                        spd: int) -> xr.DataArray:
    # https://github.com/brianpm/wavenumber_frequency/blob/master/wavenumber_frequency_functions.py#L192
    N, mlon = var_fft.shape
    pee = np.ones([N + 1, mlon + 1]) * -999

    n_i = int(N / 2)
    mlon_i = int(mlon / 2)

    # Create the real power spectrum pee = sqrt(real^2+imag^2)^2
    var_fft = np.abs(var_fft) ** 2    
    pee[ : n_i, : mlon_i] = var_fft[n_i : N, mlon_i : 0:-1]
    pee[n_i : , : mlon_i ] = var_fft[ : n_i + 1, mlon_i : 0 : -1]
    pee[ : n_i + 1, mlon_i : ] = var_fft[ n_i : : -1, : mlon_i + 1]
    pee[n_i + 1 : , mlon_i : ] = var_fft[N - 1: n_i - 1: -1 , : mlon_i + 1]
    return pee